# Assignment 2: Bigram Language Model and Generative Pretrained Transformer (GPT)


The objective of this assignment is to train a simplified transformer model. The primary differences between the implementation:
* tokenizer (we use a character level encoder simplicity and compute constraints)
* size (we are using 1 consumer grade gpu hosted on colab and a small dataset. in practice, the models are much larger and are trained on much more data)
* efficiency


Most modern LLMs have multiple training stages, so we won't get a model that is capable of replying to you yet. However, this is the first step towards a model like ChatGPT and Llama.




In [17]:
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from torch import nn

## Part 1: Bigram MLP for TinyShakespeare (35 points)

1a) (1 point). Create a list `chars` that contains all unique characters in `text`

1b) (2 points). Implement `encode(s: str) -> list[int]`

1c) (2 points). Implement `decode(ids: list[int]) -> str`

1d) (5 points). Create two tensors, `inputs_one_hot` and `outputs_one_hot`. Use one hot encoding. Make sure to get every consecutive pair of characters. For example, for the word 'hello', we should create the following input-output pairs
```
he
el
ll
lo
```

1e) (10 points). Implement BigramOneHotMLP, a 2 layer MLP that predicts the next token. Specifically, implement the constructor, forward, and generate. The output dimension of the first layer should be 8. Use `torch.optim`. The activation function for the first layer should be `nn.LeakyReLU()`

Note: Use the `torch.nn.function.cross_entropy` loss. Read the [docs](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) about how this loss function works. The logits are the output of a network WITHOUT an activation function applied to the last layer. There are activation functions are applied to every layer except the last.

1f) (5 points). Train the BigramOneHotMLP for 1000 steps.

1g) (5 points). Create two tensors, `input_ids` and `outputs_one_hot`. These `input_ids` will be used for the embedding layer.

1h) (5 points). Implement and train BigramEmbeddingMLP, a 2 layer mlp that predicts the next token. Specifically, implement the constructor, forward, and generate functions. The output dimension of the first layer should be 8. Use `torch.optim`.



Note: the output will look like gibberish


In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-10-08 16:06:51--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2025-10-08 16:06:51 (22.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
# For the bigram model, let's use the first 1000 characters for the data

with open('input.txt', 'r') as f:
    text = f.read()
text = text[:1000]

In [4]:
chars = sorted(list(set(text)))

def encode(s: str) -> list[int]:
    # implement
    stoi = {ch: i for i, ch in enumerate(chars)} # Create a dictionary mapping each character to its index
    return [stoi[ch] for ch in s] # Convert each character in string to its index number

def decode(ids: list[int]) -> str:
    # implement
    itos = {i: ch for i, ch in enumerate(chars)}  # Create reverse mapping: index to character
    return ''.join([itos[i] for i in ids]) # Convert each index back to its character

def create_one_hot_inputs_and_outputs() -> list[torch.tensor, torch.tensor]:
    # implement
    encoded_text = encode(text) # First encode the entire text to numbers

    inputs = []  # Create lists for input and output characters
    outputs = []

    for i in range(len(encoded_text) - 1): # For each consecutive pair (bigram)
        inputs.append(encoded_text[i])      # Current character
        outputs.append(encoded_text[i + 1]) # Next character

    input_indices = torch.tensor(inputs, dtype=torch.long) # Convert to tensors
    output_indices = torch.tensor(outputs, dtype=torch.long)

    inputs_one_hot = torch.nn.functional.one_hot(input_indices, num_classes=len(chars)).float() # Create one-hot encodings (sparse vectors)
    outputs_one_hot = torch.nn.functional.one_hot(output_indices, num_classes=len(chars)).float() # One-hot means: [0,0,1,0,0] where 1 is at the character's position

    return inputs_one_hot, outputs_one_hot

inputs_one_hot, outputs_one_hot = create_one_hot_inputs_and_outputs()

class BigramOneHotMLP(nn.Module):
    def __init__(self):
        # implement
        super().__init__()

        self.fc1 = nn.Linear(len(chars), 8) # Layer 1: input size = vocab size, output = 8
        self.activation = nn.LeakyReLU()

        self.fc2 = nn.Linear(8, len(chars)) # Layer 2: input = 8, output = vocab size

    def forward(self, x):
        # implement
        h = self.activation(self.fc1(x)) # Pass through first layer + activation
        logits = self.fc2(h) # Pass through second layer (no activation on output)
        return logits

    def generate(self, start='a', max_new_tokens=100) -> str:
        # implement
        generated = encode(start)

        # Get last character as one-hot
        x = torch.nn.functional.one_hot(
                torch.tensor(generated[-1]),
                num_classes=len(chars)
            ).float().unsqueeze(0)

        # Predict next character
        with torch.no_grad():
          logits = self.forward(x)
          probs = torch.nn.functional.softmax(logits, dim=-1)
          next_idx = torch.multinomial(probs, num_samples=1).item()

          generated.append(next_idx)
        return decode(generated)

bigram_one_hot_mlp = BigramOneHotMLP()

import torch.nn.functional as F

optimizer = torch.optim.Adam(bigram_one_hot_mlp.parameters(), lr=0.01)
batch_size = 32
# training loop
for _ in range(1000):
    # implement

    batch_indices = torch.randint(0, len(inputs_one_hot), (batch_size,)) # Randomly sample a batch of training examples
    x_batch = inputs_one_hot[batch_indices]
    y_batch = outputs_one_hot[batch_indices]

    logits = bigram_one_hot_mlp(x_batch) # Forward pass - get predictions

    y_indices = torch.argmax(y_batch, dim=1) # Calculate loss - cross_entropy needs indices, not one-hot
    loss = F.cross_entropy(logits, y_indices)

    optimizer.zero_grad() # Backward pass - calculate gradients
    loss.backward()

    optimizer.step() # Update weights

    if (_ + 1) % 200 == 0: # Optional: print progress
        print(f"Step {_ + 1}, Loss: {loss.item():.4f}")

print(bigram_one_hot_mlp.generate())

Step 200, Loss: 2.6498
Step 400, Loss: 1.7548
Step 600, Loss: 2.7000
Step 800, Loss: 2.3139
Step 1000, Loss: 2.4847
al


#One-Hot: Very wasteful! Uses 38 numbers to represent one letter.
Character 'a' -> [1,0,0,0,0,...,0]  (38 numbers, only one is 1)

Character 'b' -> [0,1,0,0,0,...,0]  (38 numbers, only one is 1)

#Embedding: Much more efficient! Uses only 8 numbers to represent each letter.
Character 'a' -> [0.2, -0.5, 0.8, 0.1, 0.3, -0.2, 0.7, 0.4]  (just 8 numbers!)

Character 'b' -> [0.1, 0.9, -0.3, 0.5, 0.2, 0.6, -0.1, 0.8]  (just 8 numbers!)

In [5]:
def create_embedding_inputs_and_outputs() -> list[torch.tensor, torch.tensor]:
    # implement
    encoded_text = encode(text) #inputs are just indices, not one-hot

    inputs = []
    outputs = []

    for i in range(len(encoded_text) - 1):
        inputs.append(encoded_text[i])
        outputs.append(encoded_text[i + 1])

    input_ids = torch.tensor(inputs, dtype=torch.long) # For embeddings, we use indices directly (not one-hot)

    output_indices = torch.tensor(outputs, dtype=torch.long) # Outputs are still one-hot (for loss calculation)
    outputs_one_hot = torch.nn.functional.one_hot(output_indices, num_classes=len(chars)).float()

    return input_ids, outputs_one_hot

input_ids, outputs_one_hot = create_embedding_inputs_and_outputs()

class BigramEmbeddingMLP(nn.Module):
    def __init__(self):
        # implement
        super().__init__()
        self.embedding = nn.Embedding(len(chars), 8) # Embedding layer converts indices to 8-dimensional vectors

        # Same as before: two linear layers
        self.fc1 = nn.Linear(8, 8)
        self.activation = nn.LeakyReLU()
        self.fc2 = nn.Linear(8, len(chars))

    def forward(self, x):
        # implement
        embedded = self.embedding(x)  # Convert indices to embeddings
        h = self.activation(self.fc1(embedded))
        logits = self.fc2(h)
        return logits

    def generate(self, start='a', max_new_tokens=100) -> str:
        # implement
        generated = encode(start)

        for _ in range(max_new_tokens):

            x = torch.tensor([generated[-1]], dtype=torch.long)# use the index directly

            with torch.no_grad():
                logits = self.forward(x)
                probs = torch.nn.functional.softmax(logits, dim=-1)
                next_idx = torch.multinomial(probs, num_samples=1).item()
                generated.append(next_idx)

        return decode(generated)

bigram_embedding_mlp = BigramEmbeddingMLP()

import torch.nn.functional as F
optimizer = torch.optim.Adam(bigram_embedding_mlp.parameters(), lr=0.01)
batch_size = 32

# training loop
for _ in range(1000):
    # implement
    batch_indices = torch.randint(0, len(input_ids), (batch_size,))
    x_batch = input_ids[batch_indices]  # Now using indices, not one-hot
    y_batch = outputs_one_hot[batch_indices]

    logits = bigram_embedding_mlp(x_batch)# Forward pass

    y_indices = torch.argmax(y_batch, dim=1)# Calculate loss
    loss = F.cross_entropy(logits, y_indices)

    optimizer.zero_grad()# Backward pass
    loss.backward()
    optimizer.step()

    if (_ + 1) % 200 == 0:
        print(f"Step {_ + 1}, Loss: {loss.item():.4f}")

print(bigram_embedding_mlp.generate())

Step 200, Loss: 2.2900
Step 400, Loss: 2.5170
Step 600, Loss: 2.4091
Step 800, Loss: 1.9118
Step 1000, Loss: 2.5848
ace h tirfd, ne pesoore bje cizeora o rs:
rouns anl:

ncourit, cis goard henure theny Cir cicotinesow


## Part 2: Generative Pretrained Transformer (65 points)

For this part, it is best to use a gpu. In the settings at the top go to Runtime -> Change Runtime Type and select T4 GPU

In [1]:
# run nvidia-smi to check gpu usage
!nvidia-smi

Wed Oct  8 19:51:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# For the gpt model, let's use the full text
import requests
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

print("Downloaded input.txt successfully!")
print(f"File size: {len(response.text)} characters")

with open('input.txt', 'r') as f:
    text = f.read()

print(f"Loaded text with {len(text)} characters")
print(f"First 200 characters:\n{text[:200]}")

Downloaded input.txt successfully!
File size: 1115394 characters
Loaded text with 1115394 characters
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


Implement a character level tokenization function.

1. Create a list of unique characters in the string. (1 points)
2. Implement a function `encode(s: str) -> list[int]` that takes a string and returns a list of ids (1 point)
3. Implement a function `decode(ids: list[int]) -> str` that takes a list of ids (ints) and returns a string (1 point)


In [6]:
chars = sorted(list(set(text)))  # Get all unique characters and sort them

def encode(s: str) -> list[int]:
    stoi = {ch: i for i, ch in enumerate(chars)}  # char to index mapping
    return [stoi[ch] for ch in s]

def decode(ids: list[int]) -> str:
    itos = {i: ch for i, ch in enumerate(chars)}  # index to char mapping
    return ''.join([itos[i] for i in ids])

In [9]:
import torch
data = torch.tensor(encode(text), dtype=torch.long).cuda()

In [10]:
block_size = 16
data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43],
       device='cuda:0')

To train a transformer, we feed the model `n` tokens (context) and try to predict the `n+1`th token (target) in the sequence.



In [12]:
x = data[:block_size]
y = data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18], device='cuda:0') the target: 47
when input is tensor([18, 47], device='cuda:0') the target: 56
when input is tensor([18, 47, 56], device='cuda:0') the target: 57
when input is tensor([18, 47, 56, 57], device='cuda:0') the target: 58
when input is tensor([18, 47, 56, 57, 58], device='cuda:0') the target: 1
when input is tensor([18, 47, 56, 57, 58,  1], device='cuda:0') the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15], device='cuda:0') the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47], device='cuda:0') the target: 58
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58], device='cuda:0') the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47], device='cuda:0') the target: 64
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64], device='cuda:0') the target: 43
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43], device='cuda:0') the target: 52
when input is tensor([18, 47,

In [15]:
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


### Single Self Attention Head (5 points)
![](https://i.ibb.co/GWR1XG0/head.png)

In [24]:
class SelfAttentionHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        n_embd = 128

        # Create three linear projections
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Create a lower triangular matrix for masking future tokens
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # Dropout for regularization
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape

        # Generate queries, keys, values
        q = self.query(x)  # (B, T, head_size)
        k = self.key(x)    # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # Compute attention scores ("how much should I pay attention to each word?")
        scores = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)  # scale by 1/sqrt(head_size)

        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Apply mask to prevent looking at future tokens

        # Apply softmax to get probabilities
        att_weights = F.softmax(scores, dim=-1)  # (B, T, T)
        att_weights = self.dropout(att_weights)

        # Weighted aggregation of values
        out = att_weights @ v  # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)

        return out

### Multihead Self Attention (5 points)

`constructor`

- Create 4 `SelfAttentionHead` instances. Consider using `nn.ModuleList`
- Create a linear layer with n_embd input dim and n_embd output dim

`forward`

In the forward implementation, pass `x` through each head, then concatenate all the outputs along the feature dimension, then pass the concatenated output through the linear layer

![](https://i.ibb.co/y5SwyZZ/multihead.png)

In [25]:
n_embd = 128  # embedding dimension
n_head = 4    # number of attention heads
block_size = 16  # already defined in your code
dropout = 0.1

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([SelfAttentionHead(head_size) for _ in range(num_heads)])# Create 4 attention heads
        # Linear layer to project concatenated outputs
        self.proj = nn.Linear(num_heads * head_size, n_embd)  # n_embd should be defined
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)# Pass input through each head and concatenate results
        out = self.dropout(self.proj(out))# Project back to original dimension
        return out

## MLP (2 points)
Implement a 2 layer MLP


![](https://i.ibb.co/C0DtrF5/ff.png)

In [27]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)# First layer: expand dimension by 4x
        self.fc2 = nn.Linear(4 * n_embd, n_embd) # Second layer: project back to original dimension
        self.relu = nn.ReLU()   # Activation function
        self.dropout = nn.Dropout(dropout)# Dropout for regularization

    def forward(self, x):
        x = self.fc1(x)       # Apply first linear layer  # (B, T, n_embd) -> (B, T, 4*n_embd)
        x = self.relu(x)      # ReLU activation
        x = self.fc2(x)       # Apply second linear layer # (B, T, 4*n_embd) -> (B, T, n_embd)
        x = self.dropout(x)   # Apply dropout             # Regularization
        return x

## Transformer block (20 points)

Layer normalization help training stability by normalizing the outputs of neurons within a single layer across all features for each individual data point, not across a full batch or a specific feature.

Dropout is a form of regularization to prevent overfitting.

This is the diagram of a transformer block:

![](https://i.ibb.co/X85C473/block.png)

In [28]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) # Multi-head self-attention
        self.ffwd = MLP() # Feed-forward network

        # Layer normalizations
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Attention with residual connection
        x = x + self.sa(self.ln1(x))

        # MLP with residual connection
        x = x + self.ffwd(self.ln2(x))

        return x

## GPT

`constructor` (5 points)

1. create the token embedding table and the position embedding table
2. create variable `self.blocks` that is a series of 4 `Block`s. The data will pass through each block sequentially. Consider using `nn.Sequential`
3. create a layer norm layer
4. create a linear layer for predicting the next token

`forward(self, idx, targets=None)`. (5 points)

`forward` takes a batch of context ids as input of size (B, T) and returns the logits and the loss, if targets is not None. If targets is None, return the logits and None.
1. get the token by using the token embedding table created in the constructor
2. create the position embeddings
3. sum the token and position embeddings to get the model input
4. pass the model through the blocks, the layernorm layer, and the final linear layer
5. compute the loss

`generate(start_char, max_new_tokens, top_p, top_k, temperature) -> str` (5 points)
1. implement top p, top_k, and temperature for sampling



![](https://i.ibb.co/n8sbQ0V/Screenshot-2024-01-23-at-8-59-08-PM.png)

In [29]:
class GPT(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # Token embedding table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Position embedding table
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Stack of 4 transformer blocks
        self.blocks = nn.Sequential(
            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head)
        )
        # Final layer norm
        self.ln_f = nn.LayerNorm(n_embd)
        # Language modeling head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Get token embeddings
        tok_emb = self.token_embedding_table(idx)  # (B, T, n_embd)
        # Get position embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)  # (T)
        pos_emb = self.position_embedding_table(pos)  # (T, n_embd)
        # Add token and position embeddings
        x = tok_emb + pos_emb  # (B, T, n_embd)

        # Pass through transformer blocks
        x = self.blocks(x)  # (B, T, n_embd)
        # Apply final layer norm
        x = self.ln_f(x)  # (B, T, n_embd)
        # Get logits
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            # Reshape for loss calculation
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, start_char, max_new_tokens, top_p=0.9, top_k=40, temperature=1.0):
        # Convert start character to index
        idx = torch.tensor(encode(start_char), dtype=torch.long).unsqueeze(0).to(device)

        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, _ = self.forward(idx_cond)
            # Focus on last time step
            logits = logits[:, -1, :] / temperature

            # Apply top-k filtering
            if top_k > 0:
                values, indices = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = float('-inf')

            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # Apply top-p (nucleus) filtering
            if top_p < 1.0:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                # Remove tokens with cumulative probability above threshold
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                probs[indices_to_remove] = 0.0
                probs = probs / probs.sum(dim=-1, keepdim=True)

            # Sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append to sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return decode(idx[0].tolist())

### Training loop (15 points)

implement training loop

In [31]:
# Define all necessary variables
chars = sorted(list(set(text)))  # If not already defined
vocab_size = len(chars)  # This is what's missing!
n_embd = 128
n_head = 4
block_size = 16  # You already have this
dropout = 0.1

print(f"vocab_size: {vocab_size}")
print(f"n_embd: {n_embd}")
print(f"n_head: {n_head}")

# Now create the model
model = GPT(n_embd, n_head).to('cuda')
max_iters = 5000

# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Training loop
print("Starting training...")
for iter in range(max_iters):
    # Sample a batch of data
    xb, yb = get_batch()

    # Forward pass and calculate loss
    logits, loss = model(xb, yb)

    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Print progress
    if iter % 500 == 0:
        print(f"Step {iter}: loss {loss.item():.4f}")

print("Training complete!")

vocab_size: 65
n_embd: 128
n_head: 4
Starting training...
Step 0: loss 4.3803
Step 500: loss 2.2477
Step 1000: loss 2.0035
Step 1500: loss 1.8837
Step 2000: loss 1.8405
Step 2500: loss 1.8596
Step 3000: loss 1.8254
Step 3500: loss 1.7972
Step 4000: loss 1.7798
Step 4500: loss 1.7397
Training complete!


### Generate text


print some text that your model generates

In [32]:
# Generate text samples with different settings
print("="*50)
print("Generated Text Samples")
print("="*50)

# Generate with default settings
print("\n1. Standard generation (temperature=1.0):")
generated_text = model.generate('T', max_new_tokens=200, temperature=1.0)
print(generated_text)

print("\n2. Conservative generation (temperature=0.7):")
generated_text = model.generate('W', max_new_tokens=200, temperature=0.7)
print(generated_text)

print("\n3. Creative generation (temperature=1.3):")
generated_text = model.generate('O', max_new_tokens=200, temperature=1.3)
print(generated_text)

print("\n4. With top-k and top-p filtering:")
generated_text = model.generate('I', max_new_tokens=200, top_k=40, top_p=0.9, temperature=0.9)
print(generated_text)

Generated Text Samples

1. Standard generation (temperature=1.0):
Tace him set in them
inders your for heart
The starews of not? and thee.

TESBY:
Why death, as your have lasdies, by not to be stress? her is deest thy not the preposer:
She no the grave not.

GREMIO:


2. Conservative generation (temperature=0.7):
What you have my love not with this childerer and with the trands.

SEBASTIAN:
The some so the breating the sent a visture to he cannot the world, there's land be the pleass my took
The too serrving it

3. Creative generation (temperature=1.3):
O: assiling clant though
This litcall. Loudg not this sun'n light not back!
If heard dopy, inscrisg.

Pard, ist sorthou knemenders so you are decous way: grace if slake,
Tust you ceause
Mish it you, be

4. With top-k and top-p filtering:
IS:
Will the more of such your patient:

LUCIO:
She is heart you my bust.
That be out there with our not
Warwing is more in himself?

CAMILLONT:
Sir, my must lest
In such on the prition ferst.

KING RI
